In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Bibliotecas de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    confusion_matrix, 
    classification_report,
    roc_curve
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

MODEL_DATA_PATH = Path('../../data/model_input')
RESULT_PATH = Path('../../data/results')
RESULT_PATH.mkdir(parents=True, exist_ok=True)

print("Ambiente de modelagem configurado.")

Ambiente de modelagem configurado.


## Carregar os dados 

In [3]:
print("Carregando dados de treino e teste...")

try:
    # Carregar Features (X)
    X_train = pd.read_parquet(MODEL_DATA_PATH / 'X_train.parquet')
    X_test = pd.read_parquet(MODEL_DATA_PATH / 'X_test.parquet')

    # Carregar Targets (y) - flattening para array 1D
    y_train = pd.read_csv(MODEL_DATA_PATH / 'y_train.csv').values.ravel()
    y_test = pd.read_csv(MODEL_DATA_PATH / 'y_test.csv').values.ravel()

    print(f'Treino: {X_train.shape}, Teste: {X_test.shape}')
    print(f'Prevalencia da classe positiva no treino: {y_train.mean():.2%}')
    print(f'Prevalencia da classe positiva no teste: {y_test.mean():.2%}')

except Exception as e:
    print(f'Falha ao carregar dados: {e}')

Carregando dados de treino e teste...
Treino: (11506, 18), Teste: (5277, 18)
Prevalencia da classe positiva no treino: 60.34%
Prevalencia da classe positiva no teste: 68.68%


## Função de treinamento
essa função vai evitar repetir código. Ela treina, preve e calcula todas as métricas

In [5]:
# Dicionario para armazenar resultados comparativos
resultados = []

def avaliar_modelo(nome_modelo, modelo, X_tr, y_tr, X_te, y_te):
    print(f"\n=== Treinando: {nome_modelo} ===")
    
    # 1. Treinamento
    modelo.fit(X_tr, y_tr)
    
    # 2. Predicoes (Classe e Probabilidade)
    y_pred = modelo.predict(X_te)
    try:
        y_proba = modelo.predict_proba(X_te)[:, 1]
    except:
        y_proba = np.zeros(len(y_te)) # Fallback para modelos sem probabilidade

    # 3. Calculo de Metricas
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred) # Sensibilidade (Importante para Saude)
    f1 = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_proba)
    
    print(f"Acuracia: {acc:.4f} | F1-Score: {f1:.4f} | AUC: {auc:.4f}")
    print(f"Recall (Sensibilidade): {rec:.4f}")
    
    # 4. Armazenar
    resultados.append({
        'Modelo': nome_modelo,
        'Acuracia': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AUC-ROC': auc,
        'Objeto': modelo # Guarda o modelo treinado para usar depois
    })
    
    return y_pred, y_proba

print("Funcao de avaliacao pronta.")

Funcao de avaliacao pronta.


## Execução dos Modelos 

Estou rodando e modelos: Regressão Logística, Random Forest e XGBoost (com parâmetros padrão)

In [6]:
# 1. Regressao Logistica (Baseline Linear)
# max_iter aumentado para garantir convergencia
log_reg = LogisticRegression(max_iter=1000, random_state=42)
_, _ = avaliar_modelo("Regressao Logistica", log_reg, X_train, y_train, X_test, y_test)

# 2. Random Forest (Ensemble Bagging)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
_, _ = avaliar_modelo("Random Forest", rf_clf, X_train, y_train, X_test, y_test)

# 3. XGBoost (Ensemble Boosting)
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)
_, _ = avaliar_modelo("XGBoost", xgb_clf, X_train, y_train, X_test, y_test)

# Exibir Tabela Comparativa Parcial
df_res = pd.DataFrame(resultados)
print("\n=== Comparativo Inicial ===")
print(df_res[['Modelo', 'Recall', 'F1-Score', 'AUC-ROC']].sort_values('AUC-ROC', ascending=False))


=== Treinando: Regressao Logistica ===
Acuracia: 0.7243 | F1-Score: 0.8142 | AUC: 0.7146
Recall (Sensibilidade): 0.8800

=== Treinando: Random Forest ===
Acuracia: 0.7449 | F1-Score: 0.8164 | AUC: 0.7789
Recall (Sensibilidade): 0.8259

=== Treinando: XGBoost ===
Acuracia: 0.7264 | F1-Score: 0.8043 | AUC: 0.7599
Recall (Sensibilidade): 0.8187

=== Comparativo Inicial ===
                Modelo    Recall  F1-Score   AUC-ROC
1        Random Forest  0.825883  0.816421  0.778931
2              XGBoost  0.818709  0.804283  0.759879
0  Regressao Logistica  0.879967  0.814247  0.714586


## Otimização de Hiperparâmetros

In [7]:
print("\n=== Otimizacao de Hiperparametros (XGBoost) ===")

# Definicao do espaco de busca
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Configurar Randomized Search (mais rapido que Grid Search)
# cv=3 para validacao cruzada (divisao interna do treino)
xgb_base = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    xgb_base, 
    param_distributions=param_grid, 
    n_iter=10, # Testa 10 combinacoes aleatorias
    scoring='roc_auc', 
    cv=3, 
    verbose=1, 
    random_state=42,
    n_jobs=-1
)

# Executar a busca 
random_search.fit(X_train, y_train)

print(f"\n[INFO] Melhores parametros encontrados: {random_search.best_params_}")
print(f"[INFO] Melhor AUC na validacao cruzada: {random_search.best_score_:.4f}")

# Avaliar o modelo otimizado no teste final
best_xgb = random_search.best_estimator_
_, _ = avaliar_modelo("XGBoost Tuned", best_xgb, X_train, y_train, X_test, y_test)


=== Otimizacao de Hiperparametros (XGBoost) ===
Fitting 3 folds for each of 10 candidates, totalling 30 fits

[INFO] Melhores parametros encontrados: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
[INFO] Melhor AUC na validacao cruzada: 0.6143

=== Treinando: XGBoost Tuned ===
Acuracia: 0.7535 | F1-Score: 0.8408 | AUC: 0.7511
Recall (Sensibilidade): 0.9481


## Curva ROC e Matriz de Confusão 

In [ ]:
df_resultados_final = pd.DataFrame(resultados).sort_values('AUC-ROC', ascending=False)
print("\n=== Ranking Final dos Modelos ===")
print(df_resultados_final)

# Salvar metricas em CSV
df_resultados_final.drop(columns=['Objeto']).to_csv(RESULT_PATH / 'metricas_modelos.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Curvas ROC Comparativas
axes[0].plot([0, 1], [0, 1], 'k--', label='Aleatorio')

for modelo_dict in resultados:
    nome = modelo_dict['Modelo']
    modelo = modelo_dict['Objeto']
    
    if hasattr(modelo, "predict_proba"):
        y_prob = modelo.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        axes[0].plot(fpr, tpr, label=f'{nome} (AUC = {auc:.3f})')

axes[0].set_title('Comparacao de Curvas ROC')
axes[0].set_xlabel('Taxa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos (Recall)')
axes[0].legend()

# 2. Matriz de Confusao do Campeao
melhor_modelo = resultados[-1]['Objeto'] # Pega o ultimo (XGBoost Tuned)
nome_melhor = resultados[-1]['Modelo']
y_pred_final = melhor_modelo.predict(X_test)

cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title(f'Matriz de Confusao ({nome_melhor})')
axes[1].set_xlabel('Previsto')
axes[1].set_ylabel('Real')
axes[1].set_xticklabels(['Baixo Risco', 'Alto Risco'])
axes[1].set_yticklabels(['Baixo Risco', 'Alto Risco'])

plt.tight_layout()
plt.savefig(RESULT_PATH / 'performance_modelos.png')
plt.show()